In [2]:
from pathlib import Path
import os

import clickhouse_connect

PROJECT_ROOT = Path("/home/iu31101987/mlops-final/ecom_raw_start-main")
ENV_FILE = PROJECT_ROOT / "infrastructure/clickhouse/.env"

# Читаем пароль из локального файла
password = next(
    line.split("=", 1)[1].strip()
    for line in ENV_FILE.read_text().splitlines()
    if line.startswith("CLICKHOUSE_PASSWORD=")
)

client = clickhouse_connect.get_client(
    host="127.0.0.1",
    port=8123,
    username="mlops",
    password=password,
    database="ecommerce",
)

print("Подключение успешно!")
print("Версия ClickHouse:", client.command("SELECT version()"))

Подключение успешно!
Версия ClickHouse: 25.8.33.6


In [2]:
import pandas as pd

# Загружаем категории из CSV
categories = pd.read_csv(
    PROJECT_ROOT / "category_tree.csv"
)

print("Строк в CSV:", len(categories))

# Преобразуем данные в типы, совместимые с ClickHouse
categories["categoryid"] = categories["categoryid"].astype("uint32")
categories["parentid"] = categories["parentid"].astype("Int64")

# Загружаем данные в ClickHouse
# Очищаем таблицу, чтобы повторный запуск не создавал дубликаты
client.command("TRUNCATE TABLE ecommerce.category_tree")

client.insert_df(
    "category_tree",
    categories,
)

print("Данные загружены!")

# Проверяем количество строк
count = client.command(
    "SELECT count() FROM ecommerce.category_tree"
)

print("Строк в ClickHouse:", count)

assert count == len(categories), "Количество строк не совпадает!"

Строк в CSV: 1669
Данные загружены!
Строк в ClickHouse: 1669


In [3]:
import pandas as pd

events_path = PROJECT_ROOT / "events.csv"

# Убеждаемся, что таблица пока пустая
existing_count = client.command(
    "SELECT count() FROM ecommerce.events_raw"
)

print("Сейчас строк в ClickHouse:", existing_count)

if existing_count != 0:
    raise RuntimeError(
        "Таблица events_raw не пустая. "
        "Останови загрузку, чтобы не создать дубликаты."
    )

total_loaded = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(events_path, chunksize=100_000),
    start=1
):
    # Преобразуем время из миллисекунд Unix
    chunk["event_time"] = pd.to_datetime(
        chunk["timestamp"],
        unit="ms",
        utc=True
    )

    # Сохраняем NULL для событий без транзакции
    chunk["transactionid"] = (
        chunk["transactionid"].astype("Int64")
    )

    client.insert_df(
        "events_raw",
        chunk[
            [
                "timestamp",
                "visitorid",
                "event",
                "itemid",
                "transactionid",
                "event_time",
            ]
        ],
    )

    total_loaded += len(chunk)
    print(f"Порция {chunk_number}: всего загружено {total_loaded:,}")

print("\nЗагрузка завершена!")

db_count = client.command(
    "SELECT count() FROM ecommerce.events_raw"
)

print("Строк в CSV:", total_loaded)
print("Строк в ClickHouse:", db_count)

assert total_loaded == 2_756_101
assert db_count == total_loaded

Сейчас строк в ClickHouse: 0
Порция 1: всего загружено 100,000
Порция 2: всего загружено 200,000
Порция 3: всего загружено 300,000
Порция 4: всего загружено 400,000
Порция 5: всего загружено 500,000
Порция 6: всего загружено 600,000
Порция 7: всего загружено 700,000
Порция 8: всего загружено 800,000
Порция 9: всего загружено 900,000
Порция 10: всего загружено 1,000,000
Порция 11: всего загружено 1,100,000
Порция 12: всего загружено 1,200,000
Порция 13: всего загружено 1,300,000
Порция 14: всего загружено 1,400,000
Порция 15: всего загружено 1,500,000
Порция 16: всего загружено 1,600,000
Порция 17: всего загружено 1,700,000
Порция 18: всего загружено 1,800,000
Порция 19: всего загружено 1,900,000
Порция 20: всего загружено 2,000,000
Порция 21: всего загружено 2,100,000
Порция 22: всего загружено 2,200,000
Порция 23: всего загружено 2,300,000
Порция 24: всего загружено 2,400,000
Порция 25: всего загружено 2,500,000
Порция 26: всего загружено 2,600,000
Порция 27: всего загружено 2,700,000

In [4]:
import pandas as pd

property_files = [
    PROJECT_ROOT / "item_properties_part1.csv",
    PROJECT_ROOT / "item_properties_part2.csv",
]

existing_count = client.command(
    "SELECT count() FROM ecommerce.item_properties_raw"
)

print("Сейчас строк в ClickHouse:", existing_count)

if existing_count != 0:
    raise RuntimeError(
        "Таблица item_properties_raw не пустая. "
        "Повторная загрузка может создать дубликаты."
    )

total_loaded = 0

for file_path in property_files:
    print(f"\nЗагружаем {file_path.name}")

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            file_path,
            chunksize=100_000,
            dtype={"property": "string", "value": "string"},
        ),
        start=1
    ):
        chunk["property_time"] = pd.to_datetime(
            chunk["timestamp"],
            unit="ms",
            utc=True,
        )

        client.insert_df(
            "item_properties_raw",
            chunk[
                [
                    "timestamp",
                    "itemid",
                    "property",
                    "value",
                    "property_time",
                ]
            ],
        )

        total_loaded += len(chunk)

        if chunk_number % 10 == 0:
            print(f"Всего загружено: {total_loaded:,}")

print("\nЗагрузка завершена!")

db_count = client.command(
    "SELECT count() FROM ecommerce.item_properties_raw"
)

print("Строк в CSV:", total_loaded)
print("Строк в ClickHouse:", db_count)

assert total_loaded == 20_275_902
assert db_count == total_loaded

Сейчас строк в ClickHouse: 0

Загружаем item_properties_part1.csv
Всего загружено: 1,000,000
Всего загружено: 2,000,000
Всего загружено: 3,000,000
Всего загружено: 4,000,000
Всего загружено: 5,000,000
Всего загружено: 6,000,000
Всего загружено: 7,000,000
Всего загружено: 8,000,000
Всего загружено: 9,000,000
Всего загружено: 10,000,000
Всего загружено: 10,999,999

Загружаем item_properties_part2.csv
Всего загружено: 11,999,999
Всего загружено: 12,999,999
Всего загружено: 13,999,999
Всего загружено: 14,999,999
Всего загружено: 15,999,999
Всего загружено: 16,999,999
Всего загружено: 17,999,999
Всего загружено: 18,999,999
Всего загружено: 19,999,999

Загрузка завершена!
Строк в CSV: 20275902
Строк в ClickHouse: 20275902


In [5]:
query = """
SELECT
    (SELECT count() FROM ecommerce.events_raw) AS events_count,
    (SELECT count() FROM ecommerce.item_properties_raw) AS properties_count,
    (SELECT count() FROM ecommerce.category_tree) AS categories_count,
    (SELECT countIf(event = 'view') FROM ecommerce.events_raw) AS views_count,
    (SELECT countIf(event = 'transaction') FROM ecommerce.events_raw) AS purchases_count
"""

result = client.query_df(query)

display(result)

,events_count,properties_count,categories_count,views_count,purchases_count
0,2756101,20275902,1669,2664312,22457


In [6]:
query = """
CREATE TABLE IF NOT EXISTS ecommerce.events_clean
(
    timestamp Int64,
    visitorid UInt64,
    event LowCardinality(String),
    itemid UInt64,
    transactionid Nullable(UInt64),
    event_time DateTime64(3, 'UTC')
)
ENGINE = MergeTree()
ORDER BY (itemid, event_time, visitorid)
"""

client.command(query)

print("Таблица events_clean создана")

Таблица events_clean создана


In [7]:
# Проверяем, что таблица пока пустая
count_before = client.command(
    "SELECT count() FROM ecommerce.events_clean"
)

print("Строк до загрузки:", count_before)

if count_before != 0:
    raise RuntimeError(
        "events_clean не пустая. "
        "Повторная загрузка создаст дубликаты."
    )

# Переносим уникальные события
client.command("""
INSERT INTO ecommerce.events_clean
SELECT DISTINCT
    timestamp,
    visitorid,
    event,
    itemid,
    transactionid,
    event_time
FROM ecommerce.events_raw
""")

# Проверяем результат
count_raw = client.command(
    "SELECT count() FROM ecommerce.events_raw"
)

count_clean = client.command(
    "SELECT count() FROM ecommerce.events_clean"
)

print("Исходных событий:", count_raw)
print("После очистки:", count_clean)
print("Удалено дубликатов:", count_raw - count_clean)

assert count_clean == 2_755_641
assert count_raw - count_clean == 460

print("Проверка пройдена!")

Строк до загрузки: 0
Исходных событий: 2756101
После очистки: 2755641
Удалено дубликатов: 460
Проверка пройдена!


In [8]:
query = """
SELECT
    event,
    count() AS events_count,
    min(event_time) AS first_event,
    max(event_time) AS last_event
FROM ecommerce.events_clean
GROUP BY event
ORDER BY events_count DESC
"""

display(client.query_df(query))

,event,events_count,first_event,last_event
0,view,2664218,2015-05-03 03:00:11.289,2015-09-18 02:59:47.788
1,addtocart,68966,2015-05-03 03:00:04.384,2015-09-18 02:50:49.834
2,transaction,22457,2015-05-03 03:27:21.391,2015-09-18 02:43:12.017


In [9]:
client.command("""
CREATE TABLE IF NOT EXISTS ecommerce.views_labeled
(
    timestamp Int64,
    visitorid UInt64,
    itemid UInt64,
    event_time DateTime64(3, 'UTC'),
    target UInt8
)
ENGINE = MergeTree()
ORDER BY (itemid, visitorid, event_time)
""")

print("Таблица views_labeled создана!")

Таблица views_labeled создана!


In [10]:
result = client.query_df("""
SELECT
    target,
    count() AS samples,
    round(count() * 100.0 / sum(count()) OVER (), 4) AS percent
FROM ecommerce.views_labeled
GROUP BY target
ORDER BY target
""")

display(result)

print(
    "Всего обучающих примеров:",
    client.command("SELECT count() FROM ecommerce.views_labeled")
)

""


Всего обучающих примеров: 0


In [11]:
result = client.query_df("""
SELECT
    (SELECT count()
     FROM ecommerce.events_clean
     WHERE event = 'view') AS total_views,

    (SELECT count()
     FROM ecommerce.events_clean
     WHERE event = 'view'
       AND timestamp <= (
           SELECT max(timestamp) - 86400000
           FROM ecommerce.events_clean
       )) AS eligible_views,

    (SELECT count()
     FROM ecommerce.views_labeled) AS labeled_views
""")

display(result)

,total_views,eligible_views,labeled_views
0,2664218,2654387,0


In [12]:
test_query = """
SELECT
    v.visitorid,
    v.itemid,
    v.timestamp,
    v.event_time,

    toUInt8(
        arrayExists(
            purchase_ts ->
                purchase_ts > v.timestamp
                AND purchase_ts <= v.timestamp + 86400000,
            t.purchase_times
        )
    ) AS target

FROM
(
    SELECT
        timestamp,
        visitorid,
        itemid,
        event_time
    FROM ecommerce.events_clean
    WHERE event = 'view'
      AND timestamp <= (
          SELECT max(timestamp) - 86400000
          FROM ecommerce.events_clean
      )
    LIMIT 100
) AS v

LEFT JOIN
(
    SELECT
        visitorid,
        itemid,
        groupArray(timestamp) AS purchase_times
    FROM ecommerce.events_clean
    WHERE event = 'transaction'
    GROUP BY visitorid, itemid
) AS t

ON v.visitorid = t.visitorid
AND v.itemid = t.itemid
"""

display(client.query_df(test_query).head(10))

,visitorid,itemid,timestamp,event_time,target
0,50034,245818,1438148433230,2015-07-29 05:40:33.230,0
1,188775,245818,1438291190713,2015-07-30 21:19:50.713,0
2,229809,245818,1438317355948,2015-07-31 04:35:55.948,0
3,649618,245818,1438360868615,2015-07-31 16:41:08.615,0
4,339050,245818,1438370998736,2015-07-31 19:29:58.736,0
5,351597,245822,1436934084050,2015-07-15 04:21:24.050,0
6,34327,245822,1437723034955,2015-07-24 07:30:34.955,0
7,1231193,245822,1438054184541,2015-07-28 03:29:44.541,0
8,693086,245822,1438190810739,2015-07-29 17:26:50.739,0
9,774755,245822,1438207148003,2015-07-29 21:59:08.003,0


In [15]:
result = client.query_df("""
SELECT
    event_time,
    type,
    exception_code,
    left(exception, 300) AS error_message
FROM system.query_log
WHERE event_time >= now() - INTERVAL 2 HOUR
  AND query LIKE '%INSERT INTO ecommerce.views_labeled%'
  AND type IN ('ExceptionBeforeStart', 'ExceptionWhileProcessing')
ORDER BY event_time DESC
LIMIT 5
""")

display(result)

""


In [16]:
result = client.query_df("""
SELECT
    target,
    count() AS samples
FROM
(
    SELECT
        toUInt8(
            arrayExists(
                purchase_ts ->
                    purchase_ts > v.timestamp
                    AND purchase_ts <= v.timestamp + 86400000,
                t.purchase_times
            )
        ) AS target
    FROM
    (
        SELECT timestamp, visitorid, itemid
        FROM ecommerce.events_clean
        WHERE event = 'view'
        LIMIT 100
    ) AS v
    LEFT JOIN
    (
        SELECT
            visitorid,
            itemid,
            groupArray(timestamp) AS purchase_times
        FROM ecommerce.events_clean
        WHERE event = 'transaction'
        GROUP BY visitorid, itemid
    ) AS t
    ON v.visitorid = t.visitorid
    AND v.itemid = t.itemid
)
GROUP BY target
ORDER BY target
""")

display(result)

,target,samples
0,0,95
1,1,5


In [17]:
# Проверяем, что таблица пуста
count_before = client.command(
    "SELECT count() FROM ecommerce.views_labeled"
)

print("Строк до загрузки:", count_before)

if count_before != 0:
    raise RuntimeError("Таблица уже содержит данные!")

# Загружаем просмотры за май 2015 года
client.command("""
INSERT INTO ecommerce.views_labeled
    (timestamp, visitorid, itemid, event_time, target)
SELECT
    v.timestamp,
    v.visitorid,
    v.itemid,
    v.event_time,
    toUInt8(
        arrayExists(
            purchase_ts ->
                purchase_ts > v.timestamp
                AND purchase_ts <= v.timestamp + 86400000,
            t.purchase_times
        )
    ) AS target
FROM
(
    SELECT
        timestamp,
        visitorid,
        itemid,
        event_time
    FROM ecommerce.events_clean
    WHERE event = 'view'
      AND event_time < toDateTime('2015-06-01 00:00:00', 'UTC')
      AND timestamp <= (
          SELECT max(timestamp) - 86400000
          FROM ecommerce.events_clean
      )
) AS v
LEFT JOIN
(
    SELECT
        visitorid,
        itemid,
        groupArray(timestamp) AS purchase_times
    FROM ecommerce.events_clean
    WHERE event = 'transaction'
    GROUP BY visitorid, itemid
) AS t
ON v.visitorid = t.visitorid
AND v.itemid = t.itemid
""")

print("Строк после загрузки:", client.command(
    "SELECT count() FROM ecommerce.views_labeled"
))

Строк до загрузки: 0
Строк после загрузки: 571655


In [18]:
result = client.query_df("""
SELECT
    target,
    count() AS samples,
    round(count() * 100.0 / sum(count()) OVER (), 4) AS percent
FROM ecommerce.views_labeled
GROUP BY target
ORDER BY target
""")

display(result)

print(
    "Всего строк:",
    client.command("SELECT count() FROM ecommerce.views_labeled")
)

,target,samples,percent
0,0,564355,98.723
1,1,7300,1.277


Всего строк: 571655


In [19]:
# Периоды, которые ещё не загружены
periods = [
    ("2015-06-01", "2015-07-01"),
    ("2015-07-01", "2015-08-01"),
    ("2015-08-01", "2015-09-01"),
    ("2015-09-01", "2015-10-01"),
]

for start_date, end_date in periods:

    # Защита от повторной загрузки месяца
    existing = client.query(
        """
        SELECT count()
        FROM ecommerce.views_labeled
        WHERE event_time >= toDateTime64({start: String}, 3, 'UTC')
          AND event_time < toDateTime64({end: String}, 3, 'UTC')
        """,
        parameters={"start": start_date, "end": end_date}
    ).result_rows[0][0]

    if existing > 0:
        print(f"{start_date}: уже загружено {existing:,} строк — пропускаем")
        continue

    client.command(
        """
        INSERT INTO ecommerce.views_labeled
            (timestamp, visitorid, itemid, event_time, target)
        SELECT
            v.timestamp,
            v.visitorid,
            v.itemid,
            v.event_time,
            toUInt8(
                arrayExists(
                    purchase_ts ->
                        purchase_ts > v.timestamp
                        AND purchase_ts <= v.timestamp + 86400000,
                    t.purchase_times
                )
            ) AS target
        FROM
        (
            SELECT timestamp, visitorid, itemid, event_time
            FROM ecommerce.events_clean
            WHERE event = 'view'
              AND event_time >= toDateTime64({start: String}, 3, 'UTC')
              AND event_time < toDateTime64({end: String}, 3, 'UTC')
              AND timestamp <= (
                  SELECT max(timestamp) - 86400000
                  FROM ecommerce.events_clean
              )
        ) AS v
        LEFT JOIN
        (
            SELECT
                visitorid,
                itemid,
                groupArray(timestamp) AS purchase_times
            FROM ecommerce.events_clean
            WHERE event = 'transaction'
            GROUP BY visitorid, itemid
        ) AS t
        ON v.visitorid = t.visitorid
        AND v.itemid = t.itemid
        """,
        parameters={"start": start_date, "end": end_date}
    )

    loaded = client.query(
        """
        SELECT count()
        FROM ecommerce.views_labeled
        WHERE event_time >= toDateTime64({start: String}, 3, 'UTC')
          AND event_time < toDateTime64({end: String}, 3, 'UTC')
        """,
        parameters={"start": start_date, "end": end_date}
    ).result_rows[0][0]

    print(f"{start_date}: загружено {loaded:,} строк")

print(
    "\nВсего размеченных просмотров:",
    client.command("SELECT count() FROM ecommerce.views_labeled")
)

2015-06-01: загружено 590,239 строк
2015-07-01: загружено 674,797 строк
2015-08-01: загружено 533,880 строк
2015-09-01: загружено 283,816 строк

Всего размеченных просмотров: 2654387


In [20]:
result = client.query_df("""
SELECT
    target,
    count() AS samples,
    round(
        100.0 * count() / sum(count()) OVER (),
        4
    ) AS percent
FROM ecommerce.views_labeled
GROUP BY target
ORDER BY target
""")

display(result)

total = client.command("""
    SELECT count()
    FROM ecommerce.views_labeled
""")

positives = client.command("""
    SELECT count()
    FROM ecommerce.views_labeled
    WHERE target = 1
""")

print("Всего примеров:", total)
print("Положительных примеров:", positives)
print("Отрицательных примеров:", total - positives)

,target,samples,percent
0,0,2620539,98.7248
1,1,33848,1.2752


Всего примеров: 2654387
Положительных примеров: 33848
Отрицательных примеров: 2620539


In [21]:
client.command("""
CREATE TABLE IF NOT EXISTS ecommerce.item_history
(
    itemid UInt64,
    event_time DateTime64(3, 'UTC'),
    views UInt64,
    purchases UInt64,
    ctr Float64
)
ENGINE = MergeTree()
ORDER BY (itemid, event_time)
""")

print("Таблица item_history создана!")

Таблица item_history создана!


In [22]:
query = """
SELECT
    itemid,
    event_time,
    views_at_time,
    purchases_at_time,

    sum(views_at_time) OVER (
        PARTITION BY itemid
        ORDER BY event_time
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS views,

    sum(purchases_at_time) OVER (
        PARTITION BY itemid
        ORDER BY event_time
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS purchases

FROM
(
    SELECT
        itemid,
        event_time,
        countIf(event = 'view') AS views_at_time,
        countIf(event = 'transaction') AS purchases_at_time
    FROM ecommerce.events_clean
    WHERE itemid = 245818
      AND event IN ('view', 'transaction')
    GROUP BY
        itemid,
        event_time
)
ORDER BY event_time
LIMIT 20
"""

display(client.query_df(query))

,itemid,event_time,views_at_time,purchases_at_time,views,purchases
0,245818,2015-05-03 15:14:55.674,1,0,1,0
1,245818,2015-05-04 02:24:55.554,1,0,2,0
2,245818,2015-05-04 02:26:22.942,1,0,3,0
3,245818,2015-05-04 05:05:57.376,1,0,4,0
4,245818,2015-05-04 16:09:43.915,1,0,5,0
5,245818,2015-05-04 17:47:42.858,1,0,6,0
6,245818,2015-05-05 05:12:10.256,1,0,7,0
7,245818,2015-05-05 07:30:16.212,1,0,8,0
8,245818,2015-05-05 20:40:31.933,1,0,9,0
9,245818,2015-05-05 20:48:43.903,1,0,10,0


In [23]:
query = """
SELECT
    itemid,
    event_time,
    views,
    purchases,
    round(
        if(views > 0, purchases / views, 0),
        4
    ) AS ctr
FROM
(
    SELECT
        itemid,
        event_time,

        sum(views_at_time) OVER (
            PARTITION BY itemid
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS views,

        sum(purchases_at_time) OVER (
            PARTITION BY itemid
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS purchases

    FROM
    (
        SELECT
            itemid,
            event_time,
            countIf(event = 'view') AS views_at_time,
            countIf(event = 'transaction') AS purchases_at_time
        FROM ecommerce.events_clean
        WHERE itemid = 245818
          AND event IN ('view', 'transaction')
        GROUP BY itemid, event_time
    )
)
WHERE purchases > 0
ORDER BY event_time
LIMIT 10
"""

display(client.query_df(query))

""


In [24]:
query = """
SELECT
    itemid,
    countIf(event = 'view') AS views,
    countIf(event = 'transaction') AS purchases,
    round(
        purchases / greatest(views, 1),
        4
    ) AS ctr
FROM ecommerce.events_clean
WHERE event IN ('view', 'transaction')
GROUP BY itemid
HAVING purchases > 0
ORDER BY purchases DESC
LIMIT 10
"""

display(client.query_df(query))

,itemid,views,purchases,ctr
0,461686,2538,133,0.0524
1,119736,752,97,0.1290
2,213834,293,92,0.3140
3,7943,1346,46,0.0342
4,312728,947,46,0.0486
5,445351,939,45,0.0479
6,48030,986,41,0.0416
7,420960,795,38,0.0478
8,248455,575,38,0.0661
9,17478,631,37,0.0586


In [25]:
query = """
SELECT
    itemid,
    event_time,
    views,
    purchases,
    round(
        if(views > 0, purchases / views, 0),
        4
    ) AS ctr
FROM
(
    SELECT
        itemid,
        event_time,

        sum(views_at_time) OVER (
            PARTITION BY itemid
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS views,

        sum(purchases_at_time) OVER (
            PARTITION BY itemid
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS purchases

    FROM
    (
        SELECT
            itemid,
            event_time,
            countIf(event = 'view') AS views_at_time,
            countIf(event = 'transaction') AS purchases_at_time
        FROM ecommerce.events_clean
        WHERE itemid = 461686
          AND event IN ('view', 'transaction')
        GROUP BY itemid, event_time
    )
)
WHERE purchases > 0
ORDER BY event_time
LIMIT 15
"""

display(client.query_df(query))

,itemid,event_time,views,purchases,ctr
0,461686,2015-05-04 21:03:20.263,6,1,0.1667
1,461686,2015-05-04 21:07:05.527,7,1,0.1429
2,461686,2015-05-04 22:32:07.991,8,1,0.1250
3,461686,2015-05-05 04:16:24.768,9,1,0.1111
4,461686,2015-05-05 04:17:04.367,10,1,0.1000
5,461686,2015-05-05 04:52:29.457,11,1,0.0909
6,461686,2015-05-05 05:01:37.119,12,1,0.0833
7,461686,2015-05-05 05:02:09.967,13,1,0.0769
8,461686,2015-05-05 05:03:45.520,14,1,0.0714
9,461686,2015-05-05 05:22:01.531,15,1,0.0667


In [26]:
# Проверяем состояние таблицы
existing = client.command("""
    SELECT count()
    FROM ecommerce.item_history
""")

print("Строк до загрузки:", existing)

if existing != 0:
    raise RuntimeError(
        "item_history уже содержит данные. "
        "Повторную загрузку не выполняем."
    )

# Рассчитываем накопленную историю товаров
query = """
INSERT INTO ecommerce.item_history
    (itemid, event_time, views, purchases, ctr)

SELECT
    itemid,
    event_time,
    toUInt64(views),
    toUInt64(purchases),
    if(
        views > 0,
        toFloat64(purchases) / views,
        0.0
    ) AS ctr

FROM
(
    SELECT
        itemid,
        event_time,

        sum(views_at_time) OVER (
            PARTITION BY itemid
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS views,

        sum(purchases_at_time) OVER (
            PARTITION BY itemid
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS purchases

    FROM
    (
        SELECT
            itemid,
            event_time,
            countIf(event = 'view') AS views_at_time,
            countIf(event = 'transaction') AS purchases_at_time

        FROM ecommerce.events_clean

        WHERE itemid >= 0
          AND itemid < 100000
          AND event IN ('view', 'transaction')

        GROUP BY itemid, event_time
    )
)
SETTINGS max_threads = 2
"""

client.command(query)

print("Первая группа товаров обработана!")

count = client.command("""
    SELECT count()
    FROM ecommerce.item_history
""")

print("Строк в item_history:", count)

Строк до загрузки: 0
Первая группа товаров обработана!
Строк в item_history: 566166


In [27]:
# Оставшиеся диапазоны itemid
ranges = [
    (100000, 200000),
    (200000, 300000),
    (300000, 400000),
    (400000, 500000),
]

for start_id, end_id in ranges:

    # Защита от повторной загрузки диапазона
    existing = client.query(
        """
        SELECT count()
        FROM ecommerce.item_history
        WHERE itemid >= {start: UInt64}
          AND itemid < {end: UInt64}
        """,
        parameters={
            "start": start_id,
            "end": end_id,
        }
    ).result_rows[0][0]

    if existing > 0:
        print(
            f"{start_id}–{end_id}: "
            f"уже загружено {existing:,} строк — пропускаем"
        )
        continue

    client.command(
        """
        INSERT INTO ecommerce.item_history
            (itemid, event_time, views, purchases, ctr)
        SELECT
            itemid,
            event_time,
            toUInt64(views),
            toUInt64(purchases),
            if(views > 0, toFloat64(purchases) / views, 0.0)
        FROM
        (
            SELECT
                itemid,
                event_time,
                sum(views_at_time) OVER (
                    PARTITION BY itemid
                    ORDER BY event_time
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                ) AS views,
                sum(purchases_at_time) OVER (
                    PARTITION BY itemid
                    ORDER BY event_time
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                ) AS purchases
            FROM
            (
                SELECT
                    itemid,
                    event_time,
                    countIf(event = 'view') AS views_at_time,
                    countIf(event = 'transaction') AS purchases_at_time
                FROM ecommerce.events_clean
                WHERE itemid >= {start: UInt64}
                  AND itemid < {end: UInt64}
                  AND event IN ('view', 'transaction')
                GROUP BY itemid, event_time
            )
        )
        SETTINGS max_threads = 2
        """,
        parameters={
            "start": start_id,
            "end": end_id,
        }
    )

    loaded = client.query(
        """
        SELECT count()
        FROM ecommerce.item_history
        WHERE itemid >= {start: UInt64}
          AND itemid < {end: UInt64}
        """,
        parameters={
            "start": start_id,
            "end": end_id,
        }
    ).result_rows[0][0]

    print(f"{start_id}–{end_id}: загружено {loaded:,} строк")

print(
    "\nВсего строк в item_history:",
    client.command("SELECT count() FROM ecommerce.item_history")
)

100000–200000: загружено 564,827 строк
200000–300000: загружено 590,381 строк
300000–400000: загружено 586,462 строк
400000–500000: загружено 378,816 строк

Всего строк в item_history: 2686652


In [28]:
result = client.query_df("""
SELECT
    (SELECT count()
     FROM ecommerce.item_history) AS history_rows,

    (SELECT count()
     FROM (
         SELECT itemid, event_time
         FROM ecommerce.events_clean
         WHERE event IN ('view', 'transaction')
         GROUP BY itemid, event_time
     )) AS expected_rows,

    (SELECT count()
     FROM (
         SELECT itemid, event_time
         FROM ecommerce.item_history
         GROUP BY itemid, event_time
     )) AS unique_history_rows,

    (SELECT count()
     FROM ecommerce.item_history
     WHERE views = 0 AND purchases = 0) AS empty_history_rows
""")

display(result)

,history_rows,expected_rows,unique_history_rows,empty_history_rows
0,2686652,2686652,2686652,0


In [29]:
client.command("""
CREATE TABLE IF NOT EXISTS ecommerce.views_with_history
(
    timestamp Int64,
    visitorid UInt64,
    itemid UInt64,
    event_time DateTime64(3, 'UTC'),
    target UInt8,
    views UInt64,
    purchases UInt64,
    ctr Float64
)
ENGINE = MergeTree()
ORDER BY (itemid, event_time, visitorid)
""")

print("Таблица views_with_history создана!")

Таблица views_with_history создана!


In [31]:
query = """
SELECT
    v.itemid,
    v.visitorid,
    v.event_time,
    v.target,
    if(h.itemid IS NULL, 0, h.views) AS views,
    if(h.itemid IS NULL, 0, h.purchases) AS purchases,
    if(h.itemid IS NULL, 0.0, h.ctr) AS ctr,
    h.event_time AS history_time
FROM
(
    SELECT
        timestamp,
        visitorid,
        itemid,
        event_time,
        target
    FROM ecommerce.views_labeled
    ORDER BY itemid, event_time
    LIMIT 100
) AS v
ASOF LEFT JOIN
(
    SELECT
        itemid,
        event_time,
        views,
        purchases,
        ctr
    FROM ecommerce.item_history
    ORDER BY itemid, event_time
) AS h
ON v.itemid = h.itemid
AND v.event_time > h.event_time
ORDER BY v.itemid, v.event_time
LIMIT 10
"""

display(client.query_df(query))

,itemid,visitorid,event_time,target,views,purchases,ctr,history_time
0,3,370720,2015-08-18 18:30:40.493,0,0,0,0.0,1970-01-01 00:00:00.000
1,3,639016,2015-08-31 14:39:02.792,0,1,0,0.0,2015-08-18 18:30:40.493
2,4,1042455,2015-06-30 07:03:11.545,0,0,0,0.0,1970-01-01 00:00:00.000
3,4,905555,2015-08-31 18:06:00.244,0,1,0,0.0,2015-06-30 07:03:11.545
4,4,1010132,2015-09-15 23:22:44.099,0,2,0,0.0,2015-08-31 18:06:00.244
5,6,330981,2015-05-06 20:33:13.122,0,0,0,0.0,1970-01-01 00:00:00.000
6,6,330981,2015-05-06 20:35:35.962,0,1,0,0.0,2015-05-06 20:33:13.122
7,6,275013,2015-05-20 20:39:23.871,0,2,0,0.0,2015-05-06 20:35:35.962
8,6,316532,2015-05-26 00:04:29.485,0,3,0,0.0,2015-05-20 20:39:23.871
9,6,954684,2015-05-28 10:15:05.237,0,4,0,0.0,2015-05-26 00:04:29.485


In [32]:
# Проверяем, что таблица пуста
existing = client.command(
    "SELECT count() FROM ecommerce.views_with_history"
)

print("Строк до загрузки:", existing)

if existing != 0:
    raise RuntimeError("Таблица уже содержит данные!")

# Заполняем исторические признаки для мая
client.command("""
INSERT INTO ecommerce.views_with_history
    (timestamp, visitorid, itemid, event_time,
     target, views, purchases, ctr)
SELECT
    v.timestamp,
    v.visitorid,
    v.itemid,
    v.event_time,
    v.target,
    h.views,
    h.purchases,
    h.ctr
FROM
(
    SELECT
        timestamp,
        visitorid,
        itemid,
        event_time,
        target
    FROM ecommerce.views_labeled
    WHERE event_time < toDateTime64(
        '2015-06-01 00:00:00', 3, 'UTC'
    )
    ORDER BY itemid, event_time
) AS v
ASOF LEFT JOIN
(
    SELECT
        itemid,
        event_time,
        views,
        purchases,
        ctr
    FROM ecommerce.item_history
    ORDER BY itemid, event_time
) AS h
ON v.itemid = h.itemid
AND v.event_time > h.event_time
SETTINGS max_threads = 2
""")

result = client.query_df("""
SELECT
    count() AS rows_loaded,
    countIf(views = 0) AS without_previous_views,
    countIf(purchases > 0) AS with_purchase_history
FROM ecommerce.views_with_history
""")

display(result)

Строк до загрузки: 0


,rows_loaded,without_previous_views,with_purchase_history
0,571655,114653,60200


In [33]:
periods = [
    ("2015-06-01", "2015-07-01"),
    ("2015-07-01", "2015-08-01"),
    ("2015-08-01", "2015-09-01"),
    ("2015-09-01", "2015-10-01"),
]

for start_date, end_date in periods:

    # Проверяем, есть ли уже данные за этот месяц
    existing = client.query(
        """
        SELECT count()
        FROM ecommerce.views_with_history
        WHERE event_time >= toDateTime64({start:String}, 3, 'UTC')
          AND event_time < toDateTime64({end:String}, 3, 'UTC')
        """,
        parameters={
            "start": start_date,
            "end": end_date,
        }
    ).result_rows[0][0]

    if existing > 0:
        print(f"{start_date}: уже есть {existing:,} строк — пропускаем")
        continue

    client.command(
        """
        INSERT INTO ecommerce.views_with_history
            (timestamp, visitorid, itemid, event_time,
             target, views, purchases, ctr)
        SELECT
            v.timestamp,
            v.visitorid,
            v.itemid,
            v.event_time,
            v.target,
            h.views,
            h.purchases,
            h.ctr
        FROM
        (
            SELECT
                timestamp,
                visitorid,
                itemid,
                event_time,
                target
            FROM ecommerce.views_labeled
            WHERE event_time >= toDateTime64({start:String}, 3, 'UTC')
              AND event_time < toDateTime64({end:String}, 3, 'UTC')
            ORDER BY itemid, event_time
        ) AS v
        ASOF LEFT JOIN
        (
            SELECT
                itemid,
                event_time,
                views,
                purchases,
                ctr
            FROM ecommerce.item_history
            ORDER BY itemid, event_time
        ) AS h
        ON v.itemid = h.itemid
        AND v.event_time > h.event_time
        SETTINGS max_threads = 2
        """,
        parameters={
            "start": start_date,
            "end": end_date,
        }
    )

    loaded = client.query(
        """
        SELECT count()
        FROM ecommerce.views_with_history
        WHERE event_time >= toDateTime64({start:String}, 3, 'UTC')
          AND event_time < toDateTime64({end:String}, 3, 'UTC')
        """,
        parameters={
            "start": start_date,
            "end": end_date,
        }
    ).result_rows[0][0]

    print(f"{start_date}: загружено {loaded:,} строк")

print(
    "\nВсего строк:",
    client.command(
        "SELECT count() FROM ecommerce.views_with_history"
    )
)

2015-06-01: загружено 590,239 строк
2015-07-01: загружено 674,797 строк
2015-08-01: загружено 533,880 строк
2015-09-01: загружено 283,816 строк

Всего строк: 2654387


In [34]:
result = client.query_df("""
SELECT
    property,
    count() AS total_rows,
    countDistinct(tuple(itemid, property_time)) AS unique_time_points,
    total_rows - unique_time_points AS repeated_time_points
FROM ecommerce.item_properties_raw
WHERE property IN ('categoryid', 'available')
GROUP BY property
ORDER BY property
""")

display(result)

,property,total_rows,unique_time_points,repeated_time_points
0,available,1503639,1503639,0
1,categoryid,788214,788214,0


In [3]:
query = """
SELECT
    v.itemid,
    v.event_time,
    v.views,
    v.purchases,
    v.target,
    c.categoryid,
    a.available
FROM
(
    SELECT *
    FROM ecommerce.views_with_history
    ORDER BY itemid, event_time
    LIMIT 100
) AS v
ASOF LEFT JOIN
(
    SELECT *
    FROM ecommerce.item_categories_history
    ORDER BY itemid, property_time
) AS c
ON v.itemid = c.itemid
AND v.event_time >= c.property_time
ASOF LEFT JOIN
(
    SELECT *
    FROM ecommerce.item_availability_history
    ORDER BY itemid, property_time
) AS a
ON v.itemid = a.itemid
AND v.event_time >= a.property_time
ORDER BY v.itemid, v.event_time
LIMIT 10
"""

display(client.query_df(query))

,v.itemid,event_time,views,purchases,target,categoryid,available
0,3,2015-08-18 18:30:40.493,0,0,0,1171,0
1,3,2015-08-31 14:39:02.792,1,0,0,1171,0
2,4,2015-06-30 07:03:11.545,0,0,0,1038,0
3,4,2015-08-31 18:06:00.244,1,0,0,1038,0
4,4,2015-09-15 23:22:44.099,2,0,0,1038,0
5,6,2015-05-06 20:33:13.122,0,0,0,<NA>,<NA>
6,6,2015-05-06 20:35:35.962,1,0,0,<NA>,<NA>
7,6,2015-05-20 20:39:23.871,2,0,0,1091,1
8,6,2015-05-26 00:04:29.485,3,0,0,1091,1
9,6,2015-05-28 10:15:05.237,4,0,0,1091,1


In [4]:
result = client.query_df("""
SELECT
    count() AS total_rows,
    countIf(target = 1) AS positive_samples,
    countIf(target = 0) AS negative_samples,
    countIf(categoryid IS NULL) AS missing_category,
    countIf(available IS NULL) AS missing_available,
    min(hour) AS min_hour,
    max(hour) AS max_hour,
    min(weekday) AS min_weekday,
    max(weekday) AS max_weekday
FROM ecommerce.training_dataset
""")

display(result)

,total_rows,positive_samples,negative_samples,missing_category,missing_available,min_hour,max_hour,min_weekday,max_weekday
0,2654387,33848,2620539,639598,521161,0,23,0,255


In [3]:
client.command("""
ALTER TABLE ecommerce.training_dataset
UPDATE weekday = toDayOfWeek(event_time, 1)
WHERE 1
SETTINGS mutations_sync = 2
""")

print("Дни недели исправлены!")

Дни недели исправлены!


In [4]:
display(client.query_df("""
SELECT
    min(weekday) AS min_weekday,
    max(weekday) AS max_weekday,
    countIf(weekday = 255) AS invalid_weekday
FROM ecommerce.training_dataset
"""))

,min_weekday,max_weekday,invalid_weekday
0,0,6,0
